In [1]:
import pandas as pd
import numpy as np
from scipy.stats import zscore
from gemelli.rpca import joint_rpca
from biom import Table

In [2]:
Dropbox_path='~/NYU Langone Health Dropbox/Bianca Cordazzo Vargas/Shenhav_Lab/IMiC'
joint_rpca_out_path = f'{Dropbox_path}/Code/Joint-RPCA/output/all_tps'
figure_path = f'{Dropbox_path}/Figures/Joint-RPCA'

## Load data and transform

In [3]:
#load metadata
misame_metadata = pd.read_csv(f"{joint_rpca_out_path}/misame_metadata_vitae.csv", index_col=0)

#load targeted metabolomics data
allen_macro = pd.read_csv(f"{joint_rpca_out_path}/misame_macro_all_raw.csv", index_col=0)
allen_micro = pd.read_csv(f"{joint_rpca_out_path}/misame_micro_vitae_raw.csv", index_col=0)
bode_hmos = pd.read_csv(f"{joint_rpca_out_path}/misame_hmo_all_raw.csv", index_col=0)
biocrates = pd.read_csv(f"{joint_rpca_out_path}/misame_biocrates_all_raw.csv", index_col=0)

#load top 104 untargeted metabolites
untargeted = pd.read_csv(f"{joint_rpca_out_path}/misame_untargeted_sapient.csv", index_col=0)
untargeted = untargeted.loc[misame_metadata.index]

In [4]:
#apply transformations
#targeted metabolomics data
allen_macro_tf = allen_macro.apply(np.sqrt, axis=0) #power transformation
allen_macro_tf = allen_macro_tf.apply(zscore, axis=0) #centering

allen_micro_tf = allen_micro.apply(np.sqrt, axis=0)
allen_micro_tf = allen_micro_tf.apply(zscore, axis=0)

bode_hmos_tf = bode_hmos.apply(np.sqrt, axis=0)
bode_hmos_tf = bode_hmos_tf.apply(zscore, axis=0)

biocrates_tf = biocrates.apply(np.sqrt, axis=0)
biocrates_tf = biocrates_tf.apply(zscore, axis=0)

#untargeted metabolomics data
untargeted_tf = untargeted.apply(np.sqrt, axis=1)
untargeted_tf = untargeted_tf.apply(zscore, axis=1)

In [5]:
#conver to biom tables
allen_macro_tf = Table(allen_macro_tf.values.T, 
                        observation_ids=allen_macro_tf.columns.to_list(), 
                        sample_ids=allen_macro_tf.index.to_list())
allen_micro_tf = Table(allen_micro_tf.values.T, 
                        observation_ids=allen_micro_tf.columns.to_list(), 
                        sample_ids=allen_micro_tf.index.to_list())
bode_hmos_tf = Table(bode_hmos_tf.values.T, 
                     observation_ids=bode_hmos_tf.columns.to_list(), 
                     sample_ids=bode_hmos_tf.index.to_list())
biocrates_tf = Table(biocrates_tf.values.T, 
                     observation_ids=biocrates_tf.columns.to_list(), 
                     sample_ids=biocrates_tf.index.to_list())
untargeted_tf = Table(untargeted_tf.values.T,
                      observation_ids=untargeted_tf.columns.to_list(),
                      sample_ids=untargeted_tf.index.to_list())

In [6]:
#print shape of all tables
print(misame_metadata.shape)
print(allen_macro_tf.shape)
print(allen_micro_tf.shape)
print(bode_hmos_tf.shape)
print(biocrates_tf.shape)
print(untargeted_tf.shape)

(816, 284)
(4, 816)
(23, 816)
(21, 816)
(463, 816)
(104, 816)


In [7]:
# include all modalities
ord_all, dist_all, cv_all = joint_rpca([allen_micro_tf, allen_macro_tf, bode_hmos_tf, 
                                        biocrates_tf, untargeted_tf],
                                        n_components=3,
                                        max_iterations=5,
                                        min_feature_count=10,
                                        sample_metadata=misame_metadata,
                                        rclr_transform_tables=False,
                                        train_test_column=None)
print(ord_all.proportion_explained)

#save the ordination results
#ord_all.write(f'{joint_rpca_out_path}/misame_ord_with_untarg_sapient_final.txt')

PC1    0.508343
PC2    0.490444
PC3    0.001214
dtype: float64
